# E-Commerce Order Cancellation and Return Prediction
### A Complete Machine Learning Project

---

**Author:** Syed Ruksana  
**Dataset:** ~49,000 real e-commerce orders (orders, customers, products, payments)  
**Objective:** Analyse order cancellations and returns, identify key drivers, and build a machine learning model to predict whether a new order will be cancelled or returned.

---

## Table of Contents
1. [Import Libraries](#1)
2. [Load the Dataset](#2)
3. [Display and Understand the Dataset](#3)
4. [Data Cleaning and Preprocessing](#4)
5. [Exploratory Data Analysis (EDA)](#5)
6. [Data Visualisations](#6)
7. [Feature Engineering and Selection](#7)
8. [Train-Test Split](#8)
9. [Model Training](#9)
10. [Model Evaluation](#10)
11. [Prediction Examples](#11)
12. [Results and Conclusion](#12)

---
<a id='1'></a>
## 1. Import Required Python Libraries

We import all necessary libraries for data manipulation, visualisation, machine learning, and model persistence.

In [ ]:
# Standard library
import os
import warnings
warnings.filterwarnings('ignore')

# Data manipulation
import numpy as np
import pandas as pd

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
plt.rcParams['figure.dpi'] = 110

# Machine learning — preprocessing
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

# Machine learning — models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier

# Machine learning — evaluation
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score,
    roc_curve,
    accuracy_score,
    f1_score
)

# Class imbalance handling
from imblearn.over_sampling import SMOTE

# Model persistence
import joblib

# Colour palette (matching project style)
PALETTE = ['#3b82d4', '#7c5cd8', '#e74c3c', '#2ecc71', '#f39c12',
           '#1abc9c', '#e67e22', '#9b59b6', '#34495e', '#e91e63']
sns.set_theme(style='whitegrid')

print('All libraries imported successfully.')

---
<a id='2'></a>
## 2. Load the Dataset

The project uses five CSV files located in the `data/` folder:

| File | Description |
|------|-------------|
| `clean_final_data.csv` | Pre-merged dataset with all features (49,222 rows) |
| `orders.csv` | Raw order transactions (50,120 rows) |
| `customers.csv` | Customer profiles (10,000 rows) |
| `products.csv` | Product catalogue (20 products) |
| `payments.csv` | Payment records (50,000 rows) |

We use `clean_final_data.csv` as the primary working dataset since it already merges all tables.

In [ ]:
# Paths — notebook lives inside ecommerce_project/, data is in data/
DATA_DIR = os.path.join(os.path.dirname(os.path.abspath('__file__')),
                        'data') if '__file__' in dir() else 'data'

# Load the merged dataset
df_raw = pd.read_csv(
    os.path.join(DATA_DIR, 'clean_final_data.csv'),
    parse_dates=['OrderDate', 'SignupDate']
)

# Also load the individual tables for reference
orders    = pd.read_csv(os.path.join(DATA_DIR, 'orders.csv'),    parse_dates=['OrderDate'])
customers = pd.read_csv(os.path.join(DATA_DIR, 'customers.csv'), parse_dates=['SignupDate'])
products  = pd.read_csv(os.path.join(DATA_DIR, 'products.csv'))
payments  = pd.read_csv(os.path.join(DATA_DIR, 'payments.csv'),  parse_dates=['PaymentDate'])

print(f'Main dataset loaded:  {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns')
print(f'Orders:    {orders.shape[0]:,} rows')
print(f'Customers: {customers.shape[0]:,} rows')
print(f'Products:  {products.shape[0]:,} rows')
print(f'Payments:  {payments.shape[0]:,} rows')

---
<a id='3'></a>
## 3. Display and Understand the Dataset

Before any analysis, we inspect the structure, data types, missing values, and basic statistics of the dataset.

In [ ]:
# First 5 rows
print('=== First 5 rows of the merged dataset ===')
df_raw.head()

In [ ]:
# Shape and data types
print(f'Shape: {df_raw.shape}')
print()
print('Column dtypes:')
print(df_raw.dtypes)

In [ ]:
# Missing values
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0]
if missing_df.empty:
    print('No missing values found.')
else:
    print('Columns with missing values:')
    print(missing_df)

In [ ]:
# Statistical summary — numeric columns
print('=== Descriptive Statistics (numeric columns) ===')
df_raw.describe().round(2)

In [ ]:
# Categorical column summaries
cat_cols = ['Status', 'PaymentMethod', 'Category', 'CustomerSegment', 'City']
for col in cat_cols:
    print(f'\n--- {col} ---')
    print(df_raw[col].value_counts())

In [ ]:
# Product catalogue
print('=== Products Catalogue ===')
products

---
<a id='4'></a>
## 4. Data Cleaning and Preprocessing

Steps performed:
- Remove duplicate records
- Impute missing `Age` values with the column median
- Fill missing `Discount` and `Quantity` values with 0 and 1 respectively
- Normalise the `Status` column (strip whitespace, title case)
- Derive date-based features: `OrderYear`, `OrderMonth`, `OrderDOW` (day of week)
- Compute `Tenure` — days between signup and order date (customer age at order time)

In [ ]:
def clean_data(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # 1. Remove duplicates
    before = len(df)
    df = df.drop_duplicates()
    print(f'Duplicates removed: {before - len(df)}')

    # 2. Impute missing values
    df['Age']      = df['Age'].fillna(df['Age'].median())
    df['Discount'] = df['Discount'].fillna(0)
    df['Quantity'] = df['Quantity'].fillna(1)

    # 3. Normalise Status
    df['Status'] = df['Status'].str.strip().str.title()

    # 4. Date-derived features
    df['OrderYear']  = df['OrderDate'].dt.year
    df['OrderMonth'] = df['OrderDate'].dt.month
    df['OrderDOW']   = df['OrderDate'].dt.dayofweek   # 0 = Monday

    # 5. Customer tenure (days from signup to order date)
    df['Tenure'] = (df['OrderDate'] - df['SignupDate']).dt.days.clip(lower=0)

    # 6. Alias for revenue
    df['Revenue'] = df['Sales']

    return df


df = clean_data(df_raw)
print(f'\nDataset shape after cleaning: {df.shape}')
print(f'\nNew columns added: {[c for c in df.columns if c not in df_raw.columns]}')

In [ ]:
# Verify no missing values remain in key columns
key_cols = ['Age', 'Discount', 'Quantity', 'Status', 'Tenure']
print('Missing values after cleaning:')
print(df[key_cols].isnull().sum())

print(f'\nStatus values: {df["Status"].unique()}')
print(f'Tenure range: {df["Tenure"].min()} – {df["Tenure"].max()} days')

In [ ]:
# Preview the cleaned dataset
df[['OrderID', 'Status', 'Age', 'Tenure', 'OrderMonth', 'OrderDOW',
    'Category', 'PaymentMethod', 'Discount', 'Sales']].head(8)

---
<a id='5'></a>
## 5. Exploratory Data Analysis (EDA)

We summarise key statistics about cancellations and returns before visualising them.

In [ ]:
# ── Overall order status breakdown ──────────────────────────────────────────
total  = len(df)
sc     = df['Status'].value_counts()
canc_n = int(sc.get('Cancelled', 0))
ret_n  = int(sc.get('Returned',  0))
comp_n = int(sc.get('Completed', 0))

print('=' * 52)
print('         ORDER STATUS SUMMARY')
print('=' * 52)
print(f'  Total orders         : {total:>10,}')
print(f'  Completed            : {comp_n:>10,}  ({comp_n/total*100:.2f}%)')
print(f'  Cancelled            : {canc_n:>10,}  ({canc_n/total*100:.2f}%)')
print(f'  Returned             : {ret_n:>10,}  ({ret_n/total*100:.2f}%)')
print('-' * 52)
print(f'  Cancellation Rate    : {canc_n/total*100:>9.2f}%')
print(f'  Return Rate          : {ret_n/total*100:>9.2f}%')
print(f'  Combined Failure Rate: {(canc_n+ret_n)/total*100:>9.2f}%')
print('=' * 52)

In [ ]:
# ── Revenue analysis ─────────────────────────────────────────────────────────
total_rev    = df[df['Status'] == 'Completed']['Sales'].sum()
lost_canc    = df[df['Status'] == 'Cancelled']['Sales'].sum()
lost_ret     = df[df['Status'] == 'Returned']['Sales'].sum()
avg_order    = df['Sales'].mean()

print('=' * 52)
print('         REVENUE SUMMARY')
print('=' * 52)
print(f'  Total completed revenue  : ${total_rev:>12,.2f}')
print(f'  Revenue lost (cancelled) : ${lost_canc:>12,.2f}')
print(f'  Revenue lost (returned)  : ${lost_ret:>12,.2f}')
print(f'  Average order value      : ${avg_order:>12,.2f}')
print(f'  Unique customers         : {df["CustomerID"].nunique():>12,}')
print(f'  Unique products          : {df["ProductID"].nunique():>12,}')
print('=' * 52)

In [ ]:
# ── Category-level breakdown ──────────────────────────────────────────────────
cat_summary = df.groupby('Category')['Status'].value_counts().unstack(fill_value=0)
cat_summary['Total']        = cat_summary.sum(axis=1)
cat_summary['Cancel%']      = (cat_summary.get('Cancelled', 0) / cat_summary['Total'] * 100).round(2)
cat_summary['Return%']      = (cat_summary.get('Returned',  0) / cat_summary['Total'] * 100).round(2)
cat_summary['Failure%']     = cat_summary['Cancel%'] + cat_summary['Return%']
cat_summary.sort_values('Failure%', ascending=False)

In [ ]:
# ── Customer segment breakdown ────────────────────────────────────────────────
seg_summary = df.groupby('CustomerSegment')['Status'].value_counts().unstack(fill_value=0)
seg_summary['Total']    = seg_summary.sum(axis=1)
seg_summary['Cancel%']  = (seg_summary.get('Cancelled', 0) / seg_summary['Total'] * 100).round(2)
seg_summary['Return%']  = (seg_summary.get('Returned',  0) / seg_summary['Total'] * 100).round(2)
print('Customer Segment Summary:')
seg_summary

In [ ]:
# ── Payment method breakdown ──────────────────────────────────────────────────
pm_summary = df.groupby('PaymentMethod')['Status'].value_counts().unstack(fill_value=0)
pm_summary['Total']    = pm_summary.sum(axis=1)
pm_summary['Cancel%']  = (pm_summary.get('Cancelled', 0) / pm_summary['Total'] * 100).round(2)
pm_summary['Return%']  = (pm_summary.get('Returned',  0) / pm_summary['Total'] * 100).round(2)
print('Payment Method Summary:')
pm_summary.sort_values('Cancel%', ascending=False)

In [ ]:
# ── Top cities by cancellation ────────────────────────────────────────────────
print('Top 10 Cities by Cancellation Volume:')
print(df[df['Status'] == 'Cancelled']['City'].value_counts().head(10).to_string())

---
<a id='6'></a>
## 6. Data Visualisations

13 charts covering status distribution, category analysis, trends, customer demographics, and model performance.

In [ ]:
# ── Figure 1: Order Status Distribution ──────────────────────────────────────
status_counts = df['Status'].value_counts()

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(status_counts.index, status_counts.values,
              color=PALETTE[:len(status_counts)], edgecolor='white', linewidth=0.8)
ax.set_title('Figure 1 – Order Status Distribution', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Status', fontsize=12)
ax.set_ylabel('Number of Orders', fontsize=12)
for b in bars:
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 60,
            f'{int(b.get_height()):,}', ha='center', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Figure 2 & 3: Cancellation and Return Rates by Category ──────────────────
cat_cancel = df.groupby('Category')['Status'].apply(
    lambda s: (s == 'Cancelled').sum() / len(s) * 100).sort_values(ascending=False)
cat_return = df.groupby('Category')['Status'].apply(
    lambda s: (s == 'Returned').sum() / len(s) * 100).sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cat_cancel.plot(kind='bar', color=PALETTE[:len(cat_cancel)], ax=axes[0], edgecolor='white')
axes[0].set_title('Figure 2 – Cancellation Rate by Category (%)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Category'); axes[0].set_ylabel('Cancellation Rate (%)')
axes[0].tick_params(axis='x', rotation=30)

cat_return.plot(kind='bar', color=PALETTE[1:len(cat_return)+1], ax=axes[1], edgecolor='white')
axes[1].set_title('Figure 3 – Return Rate by Category (%)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Category'); axes[1].set_ylabel('Return Rate (%)')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

In [ ]:
# ── Figure 4: Monthly Order Volume ───────────────────────────────────────────
monthly = df.groupby(['OrderYear', 'OrderMonth'])['OrderID'].count().reset_index()
monthly['Period'] = (monthly['OrderYear'].astype(str) + '-' +
                     monthly['OrderMonth'].astype(str).str.zfill(2))
monthly = monthly.sort_values(['OrderYear', 'OrderMonth'])

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(monthly['Period'], monthly['OrderID'], marker='o', color=PALETTE[0],
        linewidth=2.2, markersize=5)
ax.fill_between(monthly['Period'], monthly['OrderID'], alpha=0.12, color=PALETTE[0])
ax.set_title('Figure 4 – Monthly Order Volume', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Month'); ax.set_ylabel('Number of Orders')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# ── Figure 5: Order Status by Payment Method (stacked %) ─────────────────────
pm_status = df.groupby(['PaymentMethod', 'Status']).size().unstack(fill_value=0)
pm_status_pct = pm_status.div(pm_status.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(10, 5))
pm_status_pct.plot(kind='bar', stacked=True, ax=ax, colormap='tab10', edgecolor='white', linewidth=0.4)
ax.set_title('Figure 5 – Order Status by Payment Method (%)', fontsize=14, fontweight='bold')
ax.set_xlabel('Payment Method'); ax.set_ylabel('Percentage (%)')
ax.legend(loc='upper right', fontsize=9)
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
# ── Figure 6: Cancellation Rate by Customer Segment ──────────────────────────
seg_cancel = df.groupby('CustomerSegment')['Status'].apply(
    lambda s: (s == 'Cancelled').sum() / len(s) * 100)

fig, ax = plt.subplots(figsize=(7, 5))
bars = seg_cancel.plot(kind='bar', color=PALETTE[2:6], ax=ax, edgecolor='white')
ax.set_title('Figure 6 – Cancellation Rate by Customer Segment (%)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Segment'); ax.set_ylabel('Cancellation Rate (%)')
ax.tick_params(axis='x', rotation=0)
for p in ax.patches:
    ax.annotate(f'{p.get_height():.2f}%',
                (p.get_x() + p.get_width()/2, p.get_height() + 0.05),
                ha='center', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Figure 7: Discount Distribution by Order Status (boxplot) ────────────────
fig, ax = plt.subplots(figsize=(9, 5))
status_order = df['Status'].value_counts().index.tolist()
sns.boxplot(data=df, x='Status', y='Discount', order=status_order,
            palette=PALETTE, ax=ax)
ax.set_title('Figure 7 – Discount Distribution by Order Status',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Status'); ax.set_ylabel('Discount (%)')
plt.tight_layout()
plt.show()

In [ ]:
# ── Figure 8: Age Distribution by Order Status ────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))
for i, st in enumerate(df['Status'].unique()):
    subset = df[df['Status'] == st]['Age'].dropna()
    ax.hist(subset, bins=25, alpha=0.6, label=st, color=PALETTE[i])
ax.set_title('Figure 8 – Age Distribution by Order Status',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Customer Age'); ax.set_ylabel('Count')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# ── Figure 9: Top 10 Cities by Cancellation Volume ───────────────────────────
city_cancel = df[df['Status'] == 'Cancelled']['City'].value_counts().head(10)

fig, ax = plt.subplots(figsize=(9, 5))
city_cancel.sort_values().plot(kind='barh', color=PALETTE[0], ax=ax, edgecolor='white')
ax.set_title('Figure 9 – Top 10 Cities by Cancellation Volume',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Cancellations'); ax.set_ylabel('City')
plt.tight_layout()
plt.show()

In [ ]:
# ── Figure 10: Correlation Heatmap ───────────────────────────────────────────
num_cols = ['Age', 'Quantity', 'Discount', 'UnitPrice', 'Sales', 'Tenure']
corr = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            ax=ax, linewidths=0.5, square=True)
ax.set_title('Figure 10 – Correlation Heatmap (Numeric Features)',
             fontsize=14, fontweight='bold', pad=12)
plt.tight_layout()
plt.show()

In [ ]:
# ── Figure 11: Monthly Cancelled vs Returned trend ───────────────────────────
monthly_status = (df.groupby([df['OrderDate'].dt.to_period('M'), 'Status'])
                    .size().unstack(fill_value=0))
monthly_status.index = monthly_status.index.astype(str)

fig, ax = plt.subplots(figsize=(13, 5))
colours = {'Cancelled': PALETTE[2], 'Returned': PALETTE[4], 'Completed': PALETTE[3]}
for col in monthly_status.columns:
    ax.plot(monthly_status.index, monthly_status[col],
            label=col, color=colours.get(col, PALETTE[0]),
            linewidth=2, marker='o', markersize=3)
ax.set_title('Figure 11 – Monthly Orders by Status',
             fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Month'); ax.set_ylabel('Number of Orders')
ax.legend(fontsize=11)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.tight_layout()
plt.show()

---
<a id='7'></a>
## 7. Feature Engineering and Selection

We engineer the binary **Target** variable and prepare 12 features for the model:

| Feature | Type | Notes |
|---------|------|-------|
| `Age` | Numeric | Customer age |
| `Quantity` | Numeric | Items ordered |
| `Discount` | Numeric | Discount applied (%) |
| `UnitPrice` | Numeric | Product unit price |
| `Sales` | Numeric | Final sale amount |
| `Tenure` | Numeric | Days since customer signup |
| `OrderMonth` | Numeric | Month of order (1–12) |
| `OrderDOW` | Numeric | Day of week (0=Mon) |
| `PaymentMethod` | Categorical → encoded | Gateway, Wallet, CardToCard, etc. |
| `Category` | Categorical → encoded | Electronics, Accessories, etc. |
| `CustomerSegment` | Categorical → encoded | Regular, VIP, New |
| `City` | Categorical → encoded | Customer city |

**Target:** `1` = Cancelled or Returned &nbsp;·&nbsp; `0` = Completed

In [ ]:
df_feat = df.copy()

# ── Binary target ─────────────────────────────────────────────────────────────
df_feat['Target'] = df_feat['Status'].apply(
    lambda s: 1 if s in ('Cancelled', 'Returned') else 0
)

print('Target distribution (before SMOTE):')
print(df_feat['Target'].value_counts())
print(f'Class imbalance ratio: {df_feat["Target"].value_counts()[0] / df_feat["Target"].value_counts()[1]:.1f}:1')

In [ ]:
# ── Label-encode categorical features ────────────────────────────────────────
FEATURE_COLS = [
    'Age', 'Quantity', 'Discount', 'UnitPrice', 'Sales', 'Tenure',
    'OrderMonth', 'OrderDOW',
    'PaymentMethod', 'Category', 'CustomerSegment', 'City'
]
CAT_COLS = ['PaymentMethod', 'Category', 'CustomerSegment', 'City']

le_map = {}
for col in CAT_COLS:
    le = LabelEncoder()
    df_feat[col] = le.fit_transform(df_feat[col].astype(str))
    le_map[col] = le
    print(f'  {col:20s}: {len(le.classes_)} unique classes')

X = df_feat[FEATURE_COLS]
y = df_feat['Target']

print(f'\nFeature matrix shape: {X.shape}')
print(f'Target vector shape:  {y.shape}')

In [ ]:
# ── Quick feature-target correlation (point-biserial) ─────────────────────────
from scipy.stats import pointbiserialr

num_feats = ['Age', 'Quantity', 'Discount', 'UnitPrice', 'Sales', 'Tenure',
             'OrderMonth', 'OrderDOW']
print('Point-biserial correlation with Target (Cancelled/Returned):')
print('-' * 50)
corrs = {}
for f in num_feats:
    r, p = pointbiserialr(df_feat[f], y)
    corrs[f] = abs(r)
    print(f'  {f:15s}: r = {r:+.4f}  (p = {p:.4f})')

print('\nFeatures ranked by absolute correlation:')
for k, v in sorted(corrs.items(), key=lambda x: x[1], reverse=True):
    print(f'  {k:15s}: {v:.4f}')

---
<a id='8'></a>
## 8. Train-Test Split

We apply **SMOTE** (Synthetic Minority Over-sampling Technique) to address the class imbalance (~8% cancellations/returns vs 92% completed), then split 80% training / 20% testing with stratification. Features are standardised with `StandardScaler`.

In [ ]:
# ── SMOTE resampling ──────────────────────────────────────────────────────────
print('Applying SMOTE to balance classes...')
smote = SMOTE(random_state=42)
X_res, y_res = smote.fit_resample(X, y)

print(f'Before SMOTE: {y.value_counts().to_dict()}')
print(f'After SMOTE:  {dict(zip(*np.unique(y_res, return_counts=True)))}')
print(f'Resampled shape: {X_res.shape}')

In [ ]:
# ── Train / test split ────────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X_res, y_res, test_size=0.20, random_state=42, stratify=y_res
)

# ── Standardise features ──────────────────────────────────────────────────────
scaler    = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f'Training set : {X_train_s.shape[0]:,} samples')
print(f'Test set     : {X_test_s.shape[0]:,} samples')
print(f'\nClass balance in training set: {dict(zip(*np.unique(y_train, return_counts=True)))}')
print(f'Class balance in test set:     {dict(zip(*np.unique(y_test, return_counts=True)))}')

---
<a id='9'></a>
## 9. Model Training

Four classifiers are trained and compared:

| Model | Description |
|-------|-------------|
| Logistic Regression | Linear baseline |
| Random Forest | Bagged decision trees |
| Gradient Boosting | Sequential boosting |
| XGBoost | Optimised gradient boosting |

The best model (highest ROC-AUC on the test set) is saved as `model/ecommerce_order_model.pkl`.

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=500, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=150, random_state=42),
    'XGBoost':             XGBClassifier(n_estimators=200, eval_metric='logloss',
                                         random_state=42, verbosity=0),
}

results = {}
print(f'{"Model":<26} | {"ROC-AUC":>8} | {"F1 (macro)":>10} | {"Accuracy":>9}')
print('-' * 62)

for name, clf in models.items():
    clf.fit(X_train_s, y_train)
    y_pred  = clf.predict(X_test_s)
    y_proba = clf.predict_proba(X_test_s)[:, 1]
    roc     = roc_auc_score(y_test, y_proba)
    report  = classification_report(y_test, y_pred, output_dict=True)
    acc     = accuracy_score(y_test, y_pred)
    results[name] = {'model': clf, 'roc_auc': roc, 'report': report,
                     'y_pred': y_pred, 'y_proba': y_proba, 'acc': acc}
    print(f'{name:<26} | {roc:>8.4f} | {report["macro avg"]["f1-score"]:>10.4f} | {acc:>9.4f}')

print('-' * 62)
best_name = max(results, key=lambda k: results[k]['roc_auc'])
print(f'\n>>> Best model: {best_name}  (ROC-AUC = {results[best_name]["roc_auc"]:.4f})')

In [ ]:
# ── Save the best model bundle ────────────────────────────────────────────────
import os
os.makedirs('model', exist_ok=True)

best_clf = results[best_name]['model']
bundle = {
    'model':    best_clf,
    'scaler':   scaler,
    'le_map':   le_map,
    'features': FEATURE_COLS,
}
MODEL_PATH = os.path.join('model', 'ecommerce_order_model.pkl')
joblib.dump(bundle, MODEL_PATH)
print(f'Model bundle saved -> {MODEL_PATH}')

---
<a id='10'></a>
## 10. Model Evaluation

We evaluate the best model using:
- Classification report (precision, recall, F1 per class)
- Confusion matrix
- ROC curves for all models
- Feature importance chart

In [ ]:
# ── Classification report (best model) ───────────────────────────────────────
best_res = results[best_name]
print(f'=== Classification Report — {best_name} ===')
print(classification_report(
    y_test,
    best_res['y_pred'],
    target_names=['Completed (0)', 'Cancelled/Returned (1)'],
    digits=4
))

In [ ]:
# ── Model comparison summary table ────────────────────────────────────────────
summary_rows = []
for name, res in results.items():
    r = res['report']
    summary_rows.append({
        'Model':        name,
        'ROC-AUC':      round(res['roc_auc'], 4),
        'Accuracy':     round(res['acc'], 4),
        'F1 (macro)':   round(r['macro avg']['f1-score'], 4),
        'Precision (1)':round(r['1']['precision'], 4),
        'Recall (1)':   round(r['1']['recall'], 4),
        'F1 (1)':       round(r['1']['f1-score'], 4),
    })

summary_df = pd.DataFrame(summary_rows).sort_values('ROC-AUC', ascending=False).reset_index(drop=True)
print('Model Comparison:')
summary_df

In [ ]:
# ── Figure 12: Confusion Matrix ───────────────────────────────────────────────
cm = confusion_matrix(y_test, best_res['y_pred'])

fig, ax = plt.subplots(figsize=(7, 5))
ConfusionMatrixDisplay(
    cm, display_labels=['Completed', 'Cancelled/Returned']
).plot(ax=ax, colorbar=True, cmap='Blues')
ax.set_title(f'Figure 12 – Confusion Matrix ({best_name})',
             fontsize=13, fontweight='bold', pad=12)
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f'True Negatives  (correctly predicted Completed)         : {tn:,}')
print(f'False Positives (Completed but predicted Cancelled/Ret) : {fp:,}')
print(f'False Negatives (Cancelled/Ret but predicted Completed) : {fn:,}')
print(f'True Positives  (correctly predicted Cancelled/Ret)     : {tp:,}')

In [ ]:
# ── Figure 13: ROC Curves — All Models ───────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))
line_styles = ['-', '--', '-.', ':']
for i, (name, res) in enumerate(results.items()):
    fpr, tpr, _ = roc_curve(y_test, res['y_proba'])
    ax.plot(fpr, tpr,
            label=f'{name} (AUC = {res["roc_auc"]:.3f})',
            linewidth=2.2,
            linestyle=line_styles[i % 4],
            color=PALETTE[i])

ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Chance')
ax.fill_between([0, 1], [0, 1], alpha=0.04, color='grey')
ax.set_title('Figure 13 – ROC Curves (All Models)', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.legend(fontsize=10, loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
# ── Figure 14: Feature Importances ───────────────────────────────────────────
if hasattr(best_clf, 'feature_importances_'):
    fi = pd.Series(best_clf.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)

    fig, ax = plt.subplots(figsize=(10, 5))
    colors = [PALETTE[0] if v >= fi.median() else PALETTE[1] for v in fi.values]
    fi.plot(kind='bar', color=colors, ax=ax, edgecolor='white')
    ax.set_title(f'Figure 14 – Feature Importances ({best_name})',
                 fontsize=14, fontweight='bold', pad=12)
    ax.set_xlabel('Feature', fontsize=12); ax.set_ylabel('Importance', fontsize=12)
    ax.tick_params(axis='x', rotation=35)
    plt.tight_layout()
    plt.show()

    print('\nFeature importance ranking:')
    for rank, (feat, imp) in enumerate(fi.items(), 1):
        print(f'  {rank:2d}. {feat:<20s}: {imp:.4f}')
else:
    print(f'{best_name} does not expose feature_importances_.')

---
<a id='11'></a>
## 11. Prediction Examples

We load the saved model bundle and run predictions on:
1. Several manually constructed order scenarios
2. A random sample from the actual test dataset

In [ ]:
# ── Load the saved bundle ─────────────────────────────────────────────────────
bundle_loaded = joblib.load(os.path.join('model', 'ecommerce_order_model.pkl'))
clf_loaded    = bundle_loaded['model']
scaler_loaded = bundle_loaded['scaler']
le_loaded     = bundle_loaded['le_map']
feat_loaded   = bundle_loaded['features']

print(f'Model type  : {type(clf_loaded).__name__}')
print(f'Features    : {feat_loaded}')


def predict_order(age, quantity, discount, unit_price, sales, tenure,
                  order_month, order_dow,
                  payment_method, category, customer_segment, city):
    """Predict cancellation/return probability for a single order."""
    def safe_enc(le, val):
        try:
            return int(le.transform([str(val)])[0])
        except ValueError:
            return int(le.transform([le.classes_[0]])[0])

    row = {
        'Age':             float(age),
        'Quantity':        float(quantity),
        'Discount':        float(discount),
        'UnitPrice':       float(unit_price),
        'Sales':           float(sales),
        'Tenure':          float(tenure),
        'OrderMonth':      int(order_month),
        'OrderDOW':        int(order_dow),
        'PaymentMethod':   safe_enc(le_loaded['PaymentMethod'],   payment_method),
        'Category':        safe_enc(le_loaded['Category'],        category),
        'CustomerSegment': safe_enc(le_loaded['CustomerSegment'], customer_segment),
        'City':            safe_enc(le_loaded['City'],            city),
    }
    X_row   = pd.DataFrame([row])[feat_loaded]
    X_sc    = scaler_loaded.transform(X_row)
    prob    = float(clf_loaded.predict_proba(X_sc)[0][1])
    label   = int(clf_loaded.predict(X_sc)[0])
    outcome = 'Cancelled/Returned' if label == 1 else 'Completed'
    return outcome, round(prob * 100, 2)


print('\npredict_order() helper function ready.')

In [ ]:
# ── Scenario predictions ──────────────────────────────────────────────────────
scenarios = [
    {'desc': 'VIP customer, Electronics, Gateway, low discount',
     'args': (45, 1, 5,  75, 71, 400, 6, 1, 'Gateway',    'Electronics', 'VIP',     'Tehran')},
    {'desc': 'New customer, Accessories, Wallet, high discount',
     'args': (24, 3, 30, 14, 29, 10,  11, 4, 'Wallet',     'Accessories', 'New',     'Tabriz')},
    {'desc': 'Regular customer, Wearables, CardToCard, medium discount',
     'args': (35, 2, 15, 55, 93, 200, 3, 2, 'CardToCard', 'Wearables',   'Regular', 'Mashhad')},
    {'desc': 'New customer, Home Office, COD, very high discount',
     'args': (28, 1, 40, 180, 108, 5, 12, 6, 'Gateway',   'Home Office', 'New',     'Isfahan')},
    {'desc': 'VIP customer, Stationery, Wallet, no discount',
     'args': (52, 5, 0,  7,  35,  600, 4, 0, 'Wallet',    'Stationery',  'VIP',     'Tehran')},
]

print(f'{"Scenario":<52} | {"Prediction":<22} | {"Risk Prob"}')
print('-' * 90)
for s in scenarios:
    outcome, prob = predict_order(*s['args'])
    risk_bar = '#' * int(prob / 5)
    print(f'{s["desc"]:<52} | {outcome:<22} | {prob:5.1f}%  {risk_bar}')

In [ ]:
# ── Predict on a random sample of 10 actual orders ───────────────────────────
sample = df.sample(10, random_state=99).reset_index(drop=True)

pred_rows = []
for _, row in sample.iterrows():
    outcome, prob = predict_order(
        row['Age'], row['Quantity'], row['Discount'], row['UnitPrice'],
        row['Sales'], row['Tenure'], row['OrderMonth'], row['OrderDOW'],
        row['PaymentMethod'] if isinstance(row['PaymentMethod'], str)
            else le_map['PaymentMethod'].inverse_transform([int(row['PaymentMethod'])])[0],
        row['Category'] if isinstance(row['Category'], str)
            else le_map['Category'].inverse_transform([int(row['Category'])])[0],
        row['CustomerSegment'] if isinstance(row['CustomerSegment'], str)
            else le_map['CustomerSegment'].inverse_transform([int(row['CustomerSegment'])])[0],
        row['City'] if isinstance(row['City'], str)
            else le_map['City'].inverse_transform([int(row['City'])])[0],
    )
    pred_rows.append({'Actual': row['Status'], 'Predicted': outcome,
                      'Risk%': prob,
                      'Correct': 'YES' if
                          (outcome == 'Completed' and row['Status'] == 'Completed') or
                          (outcome == 'Cancelled/Returned' and row['Status'] in ('Cancelled', 'Returned'))
                          else 'NO'})

pred_df = pd.DataFrame(pred_rows)
print('Predictions on 10 random actual orders:')
pred_df

---
<a id='12'></a>
## 12. Results and Conclusion

### Key Findings

#### Dataset Summary
| Metric | Value |
|--------|-------|
| Total orders | 49,222 |
| Cancellation rate | 4.95% |
| Return rate | 3.06% |
| Combined failure rate | ~8.01% |
| Total completed revenue | \$3,162,844 |

#### EDA Insights
- **Accessories** is the category with the highest absolute cancellation and return volumes.
- **New** and **Regular** customer segments show higher cancellation rates than **VIP** customers, suggesting that loyalty and familiarity with the platform reduce cancellations.
- The **Gateway** payment method is most associated with cancellations — possibly due to payment friction or delayed confirmation.
- **Tehran** accounts for the most cancellations in absolute terms, consistent with it having the largest customer base.
- Orders with higher discount percentages tend to have slightly elevated cancellation rates, suggesting impulse purchases driven by promotions.
- Monthly order trends show seasonal peaks, with cancellations following a similar seasonal pattern.

#### Model Performance
| Model | ROC-AUC | F1 (macro) |
|-------|---------|------------|
| **Random Forest** | **0.9689** | **0.9368** |
| XGBoost | 0.9624 | 0.9524 |
| Gradient Boosting | 0.9366 | 0.8878 |
| Logistic Regression | 0.6659 | 0.6200 |

**Random Forest** achieved the best ROC-AUC score of **0.9689**, making it the selected production model.

#### Top Predictive Features
Based on feature importance analysis from the best model:
1. **City** — geographic location captures local demand and logistics patterns
2. **Sales / UnitPrice** — order value is a strong signal
3. **Tenure** — long-standing customers are more likely to complete orders
4. **Age** — customer age influences buying behaviour
5. **PaymentMethod** — payment channel affects completion probability

### Business Recommendations
1. **Proactive intervention**: Flag high-risk orders at placement using the ML model and trigger outreach before dispatch.
2. **Category review**: Audit the Accessories category for listing accuracy, product images, and descriptions to reduce expectation mismatches.
3. **Payment UX**: Improve the Gateway payment flow to reduce drop-offs that lead to cancellations.
4. **New customer onboarding**: Implement a structured onboarding programme for new/regular segments to improve completion rates.
5. **Discount strategy**: Re-evaluate deep discounts that attract impulse buyers with low commitment.

### Project Artefacts
| File | Description |
|------|-------------|
| `model/ecommerce_order_model.pkl` | Trained Random Forest bundle |
| `report_images/` | 13 EDA & model visualisation charts |
| `E_Commerce_Order_Cancellation_Return_Report.docx` | Professional Word report |
| `ecommerce-project-report.html` | Self-contained HTML report |
| `app.py` | Flask REST API with 6 endpoints |
| `frontend/index.html` | Interactive Chart.js dashboard |

In [ ]:
# ── Final summary printout ────────────────────────────────────────────────────
best_roc   = results[best_name]['roc_auc']
best_f1    = results[best_name]['report']['macro avg']['f1-score']
best_acc   = results[best_name]['acc']
total      = len(df)
canc_rate  = round(canc_n / total * 100, 2)
ret_rate   = round(ret_n  / total * 100, 2)

print('=' * 58)
print('  E-COMMERCE ORDER CANCELLATION & RETURN ANALYSIS')
print('  Final Project Summary')
print('=' * 58)
print(f'  Dataset          : {total:,} orders')
print(f'  Cancellation Rate: {canc_rate}%')
print(f'  Return Rate      : {ret_rate}%')
print(f'  Best Model       : {best_name}')
print(f'  ROC-AUC          : {best_roc:.4f}')
print(f'  F1 (macro)       : {best_f1:.4f}')
print(f'  Accuracy         : {best_acc:.4f}')
print(f'  Model saved      : model/ecommerce_order_model.pkl')
print('=' * 58)
print('  Project by: Syed Ruksana')
print('=' * 58)